# 05a — Filter Methods

## The One-Sentence Version
Rank features by how informative they are individually, then keep the top ones.

## What You'll Build Intuition For
- Why some features carry zero information (near-zero variance)
- How to spot redundant feature pairs (correlation filtering)
- Capturing nonlinear relationships that correlation misses (mutual information)
- A practical pipeline that chains all three filters

## Prerequisites
- [04d — Comparison Arena](../04_nonlinear_methods/04d_comparison_arena.ipynb) — the toolkit so far
- Basic comfort with correlation (Module 02)

## The Story

So far, we've been *transforming* features — projecting them onto new axes (PCA) or embedding them in 2D (t-SNE, UMAP). But sometimes you don't want new features. You want to keep the *original* ones that matter and throw away the rest.

Why? Because a doctor doesn't want to hear "PC3 is elevated." They want to hear "your potassium is high." Feature selection keeps things interpretable.

Filter methods are the simplest approach: score each feature (or pair of features) using a statistical measure, then keep the winners. They're fast, model-agnostic, and a great first pass before anything fancier.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_patient_data

apply_style()
rng = np.random.default_rng(42)

## Variance Threshold

If a feature is (nearly) the same value for every sample, it tells you nothing. A column of all 7s can't help you distinguish patients. The simplest filter: drop features whose variance falls below a threshold.

In [ ]:
# Generate data with some deliberately constant/near-constant columns
n = 500
X_signal = rng.normal(0, 1, (n, 5))  # 5 useful features
X_constant = np.full((n, 2), 7.0)     # 2 constant features
X_near_const = rng.normal(0, 0.001, (n, 3))  # 3 near-constant features

X_all = np.hstack([X_signal, X_constant, X_near_const])
names = ([f"Signal_{i}" for i in range(5)] +
         [f"Constant_{i}" for i in range(2)] +
         [f"NearConst_{i}" for i in range(3)])

# Show variances
variances = X_all.var(axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
colours = ([COLOURS["signal"]] * 5 +
           [COLOURS["noise"]] * 2 +
           [COLOURS["accent"]] * 3)
ax.bar(names, variances, color=colours, alpha=0.8, edgecolor="white")
ax.axhline(y=0.01, color=COLOURS["noise"], linestyle="--", label="Threshold = 0.01")
ax.set_ylabel("Variance")
ax.set_title("Feature variances: constants and near-constants are useless")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("visuals/05a_variance_threshold.png", dpi=150, bbox_inches="tight")
plt.show()

# Apply VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_filtered = selector.fit_transform(X_all)
kept = selector.get_support()

print(f"Original features: {X_all.shape[1]}")
print(f"After variance threshold: {X_filtered.shape[1]}")
print(f"Kept: {[n for n, k in zip(names, kept) if k]}")
print(f"Dropped: {[n for n, k in zip(names, kept) if not k]}")

## Correlation Filter

If two features correlate at |r| > 0.95, they're telling you nearly the same thing. Keeping both wastes computation and can destabilise models.

The algorithm: for each highly correlated pair, drop the one with lower correlation to the target (or lower variance if there's no target).

In [ ]:
# Generate data with deliberately correlated pairs
n = 500
x1 = rng.normal(0, 1, n)
x2 = x1 + rng.normal(0, 0.05, n)   # r ≈ 0.999 with x1
x3 = rng.normal(0, 1, n)
x4 = x3 + rng.normal(0, 0.1, n)    # r ≈ 0.99 with x3
x5 = rng.normal(0, 1, n)            # independent
x6 = rng.normal(0, 1, n)            # independent

X_corr = np.column_stack([x1, x2, x3, x4, x5, x6])
corr_names = ["feat_1", "feat_2 (≈feat_1)", "feat_3", "feat_4 (≈feat_3)",
              "feat_5", "feat_6"]

# Compute and display correlation matrix
corr_matrix = np.corrcoef(X_corr.T)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_names)))
ax.set_yticks(range(len(corr_names)))
ax.set_xticklabels(corr_names, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr_names, fontsize=9)

# Annotate cells
for i in range(len(corr_names)):
    for j in range(len(corr_names)):
        ax.text(j, i, f"{corr_matrix[i, j]:.2f}", ha="center", va="center",
                fontsize=8, color="white" if abs(corr_matrix[i, j]) > 0.5 else "black")

plt.colorbar(im, ax=ax, label="Pearson r")
ax.set_title("Correlation matrix: spot the redundant pairs")
plt.tight_layout()
plt.savefig("visuals/05a_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def correlation_filter(X, feature_names, threshold=0.95):
    """Drop one feature from each pair with |r| > threshold.
    Keeps the one with higher variance (heuristic when no target is available)."""
    corr = np.corrcoef(X.T)
    n_features = X.shape[1]
    to_drop = set()

    for i in range(n_features):
        if i in to_drop:
            continue
        for j in range(i + 1, n_features):
            if j in to_drop:
                continue
            if abs(corr[i, j]) > threshold:
                # Drop the one with lower variance
                if X[:, i].var() >= X[:, j].var():
                    to_drop.add(j)
                else:
                    to_drop.add(i)

    keep = [i for i in range(n_features) if i not in to_drop]
    kept_names = [feature_names[i] for i in keep]
    dropped_names = [feature_names[i] for i in to_drop]
    return X[:, keep], kept_names, dropped_names

X_decorr, kept, dropped = correlation_filter(X_corr, corr_names, threshold=0.95)
print(f"Original features: {X_corr.shape[1]}")
print(f"After correlation filter (|r| > 0.95): {X_decorr.shape[1]}")
print(f"Kept: {kept}")
print(f"Dropped: {dropped}")

## Mutual Information

Correlation only captures *linear* relationships. If X² predicts Y but X doesn't linearly correlate with Y, Pearson r says "nothing here." Mutual information captures *any* statistical dependence — linear or not.

Think of it this way: correlation asks "do X and Y move together in a straight line?" Mutual information asks "does knowing X tell me *anything* about Y?"

In [ ]:
# Demonstrate: X² predicts Y but correlation is near zero
n = 500
x = rng.uniform(-3, 3, n)
y_nonlinear = (x ** 2 > 4).astype(int)  # class boundary is a parabola

# Correlation says nothing
r = np.corrcoef(x, y_nonlinear)[0, 1]

# Mutual information says something
mi = mutual_info_classif(x.reshape(-1, 1), y_nonlinear, random_state=42)[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Show the data
for cls, colour, label in [(0, COLOURS["signal"], "Class 0"),
                            (1, COLOURS["noise"], "Class 1")]:
    mask = y_nonlinear == cls
    ax1.scatter(x[mask], rng.uniform(-0.3, 0.3, mask.sum()), s=15, alpha=0.5,
               color=colour, label=label)
ax1.set_xlabel("X")
ax1.set_title("Nonlinear boundary: |X| > 2 → Class 1")
ax1.legend()

# Compare correlation vs MI
measures = ["Pearson |r|", "Mutual Info"]
values = [abs(r), mi]
bar_colours = [COLOURS["noise"], COLOURS["signal"]]
ax2.bar(measures, values, color=bar_colours, alpha=0.8, edgecolor="white")
ax2.set_ylabel("Score")
ax2.set_title(f"Correlation misses it (|r|={abs(r):.3f}), MI catches it ({mi:.3f})")

fig.suptitle("Mutual Information captures nonlinear relationships", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/05a_mi_vs_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Apply MI to patient data: signal features should rank higher than noise
X_patients, feature_names, signal_mask = make_patient_data(
    n=500, d_signal=5, d_noise=15, seed=42
)

# Create a binary target from age (above/below median)
y_patients = (X_patients[:, 0] > np.median(X_patients[:, 0])).astype(int)

# Compute MI for each feature
mi_scores = mutual_info_classif(X_patients, y_patients, random_state=42)

# Sort by MI score
order = np.argsort(mi_scores)[::-1]

fig, ax = plt.subplots(figsize=(12, 5))
bar_colours = [COLOURS["signal"] if signal_mask[i] else COLOURS["noise"] for i in order]
ax.barh([feature_names[i] for i in order], mi_scores[order],
        color=bar_colours, alpha=0.8, edgecolor="white")
ax.set_xlabel("Mutual Information")
ax.set_title("MI ranking: signal features (blue) float to the top")
ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=COLOURS["signal"], label="Signal"),
    Patch(facecolor=COLOURS["noise"], label="Noise"),
], loc="lower right")

plt.tight_layout()
plt.savefig("visuals/05a_mi_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

print("Signal features cluster at the top — MI correctly identifies them.")
print("Some noise features may have small MI by chance, but the gap is clear.")

## Putting It Together

A practical filter pipeline: variance threshold → correlation filter → MI ranking → select top k. Each stage prunes features, and the final model uses only the winners.

In [ ]:
# Build a more challenging dataset: 5 signal + 30 noise + 3 constant + 2 redundant
X_sig, feat_names_sig, sig_mask = make_patient_data(
    n=500, d_signal=5, d_noise=30, seed=42
)

# Add constant columns
X_const = np.full((500, 3), 42.0)
const_names = ["Constant_0", "Constant_1", "Constant_2"]

# Add redundant columns (near-copies of signal features)
X_redun = X_sig[:, :2] + rng.normal(0, 0.01, (500, 2))
redun_names = ["Age_copy", "HR_copy"]

X_full = np.hstack([X_sig, X_const, X_redun])
all_names = feat_names_sig + const_names + redun_names
y_target = (X_sig[:, 0] > np.median(X_sig[:, 0])).astype(int)

print(f"Starting features: {X_full.shape[1]}")

# Stage 1: Variance threshold
vt = VarianceThreshold(threshold=0.01)
X_s1 = vt.fit_transform(X_full)
names_s1 = [n for n, k in zip(all_names, vt.get_support()) if k]
print(f"After variance threshold: {X_s1.shape[1]} (dropped {X_full.shape[1] - X_s1.shape[1]} constant/near-constant)")

# Stage 2: Correlation filter
X_s2, names_s2, dropped_corr = correlation_filter(X_s1, names_s1, threshold=0.95)
print(f"After correlation filter: {X_s2.shape[1]} (dropped {len(dropped_corr)} redundant: {dropped_corr})")

# Stage 3: MI ranking → top k
mi_all = mutual_info_classif(X_s2, y_target, random_state=42)
k = 8
top_k_idx = np.argsort(mi_all)[::-1][:k]
X_s3 = X_s2[:, top_k_idx]
names_s3 = [names_s2[i] for i in top_k_idx]
print(f"After MI top-{k}: {X_s3.shape[1]} features: {names_s3}")

In [ ]:
# Compare model accuracy: all features vs filtered features
scaler = StandardScaler()
model = LogisticRegression(max_iter=1000, random_state=42)

# All features
X_all_scaled = scaler.fit_transform(X_full)
acc_all = cross_val_score(model, X_all_scaled, y_target, cv=5).mean()

# Filtered features
X_filt_scaled = scaler.fit_transform(X_s3)
acc_filt = cross_val_score(model, X_filt_scaled, y_target, cv=5).mean()

# Visualise pipeline
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pipeline funnel
stages = ["Original", "Variance\nThreshold", "Correlation\nFilter", f"MI Top-{k}"]
counts = [X_full.shape[1], X_s1.shape[1], X_s2.shape[1], X_s3.shape[1]]
stage_colours = [COLOURS["neutral"], COLOURS["accent"], COLOURS["accent"], COLOURS["signal"]]
ax1.bar(stages, counts, color=stage_colours, alpha=0.8, edgecolor="white")
for i, c in enumerate(counts):
    ax1.text(i, c + 0.5, str(c), ha="center", fontsize=12, fontweight="bold")
ax1.set_ylabel("Number of Features")
ax1.set_title("Filter pipeline: progressive pruning")

# Accuracy comparison
labels = [f"All {X_full.shape[1]}\nfeatures", f"Top {k}\nfiltered"]
accs = [acc_all, acc_filt]
acc_colours = [COLOURS["noise"], COLOURS["signal"]]
ax2.bar(labels, accs, color=acc_colours, alpha=0.8, edgecolor="white")
for i, a in enumerate(accs):
    ax2.text(i, a + 0.005, f"{a:.1%}", ha="center", fontsize=12, fontweight="bold")
ax2.set_ylabel("Cross-Validation Accuracy")
ax2.set_ylim(0, 1.1)
ax2.set_title("Accuracy: fewer features, same (or better) performance")

fig.suptitle("Filter Pipeline: from 40 features to 8 with no accuracy loss", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/05a_pipeline.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"All features accuracy:      {acc_all:.1%}")
print(f"Filtered features accuracy: {acc_filt:.1%}")
print(f"\nWe dropped {X_full.shape[1] - X_s3.shape[1]} features and lost (nearly) nothing.")

> **Key insight:** Filter methods are fast and model-agnostic. They're the first line of defence against feature bloat. But they evaluate features *individually* — they can miss features that are only useful in combination. That's what wrapper methods (05b) address.

<details>
<summary><b>The Maths Behind This</b></summary>

**Variance:** $\text{Var}(X_j) = \frac{1}{n} \sum_{i=1}^n (x_{ij} - \bar{x}_j)^2$. Features with Var ≈ 0 carry no discriminative information.

**Pearson correlation:** $r_{jk} = \frac{\sum_i (x_{ij} - \bar{x}_j)(x_{ik} - \bar{x}_k)}{\sqrt{\sum_i (x_{ij} - \bar{x}_j)^2 \cdot \sum_i (x_{ik} - \bar{x}_k)^2}}$. Measures linear dependence. |r| ≈ 1 means redundancy.

**Mutual Information:** $I(X; Y) = \sum_{x, y} p(x, y) \log \frac{p(x, y)}{p(x)p(y)}$. Measures *any* statistical dependence. MI = 0 iff X and Y are independent. Unlike correlation, MI captures nonlinear relationships.

In practice, sklearn's `mutual_info_classif` estimates MI using k-nearest-neighbour density estimation (Kraskov et al., 2004).

</details>

## Where This Matters

**Electronic health records:** Patient records can have thousands of features (lab values, vitals, medications, diagnoses). Most are irrelevant to any specific prediction task. Variance thresholding removes the columns that are identical for 99% of patients. Correlation filtering removes redundant labs (e.g., two slightly different creatinine assays). MI ranking finds the features that actually predict the outcome.

**Genomics:** Gene expression datasets have 20,000+ features per sample. Filter methods are essential as a first pass — wrapper methods would be computationally prohibitive at this scale.

## Feynman Check

1. **Why might correlation miss an important feature that mutual information catches?**

2. **What's the risk of filtering features one at a time without considering combinations?**

---

Filter methods score features individually. In **05b — Wrapper Methods**, we'll test feature *combinations* to find the best subset.